In [ ]:
# 여러에이전트 협력
# 메세지 기반 통신
# coordinator를 통한 라우팅
# 3가지 특화 에이전트 (Text, Math, Date)
# 비동기화 메세지 통신 
    # ex. A,B,C agent가 있다고 가정하면, 
    # 비동기화는 A, B, C 각각이 진행됨
    # 동기화가 좋은게 아님?? 순차적으로 진행되서.. A가 완료되고나야 B가 진행됨..
    # 비동기화 --> 웹페이지 (페이지에서 구역에 따라 바로 업데이트가 되는 곳이 있고, 고정된 구역이 존재..)
    # 스트림릿은 비동기화가 안되어있음
    # Django? 비동기화 가능함


<span style="font-size:13px;">

```text

1️⃣ 상위 로직이 질문을 보고 “날씨 Agent에게 보내야겠다” 판단
    if "날씨" in user_question:
        weather_agent.send_message(
            receiver_id=weather_agent.agent_id,
            content={"location": "서울"})
📌 이 단계도 네 코드에 없음
- 질문 분석
- Agent 선택
- 분기 판단
👉 이건 Decision Router / Controller
👉 (❌ 네 코드에 없음, 나중에 구현 가능)



2️⃣ 날씨 Agent가 send_message() 호출 → Message 생성
    weather_agent.send_message(
        receiver_id=weather_agent.agent_id,
        content={"location": "서울"}
    )
✔️ 이 부분은 네 코드에 있음
실제로 일어나는 일:
- Message 객체 생성
- sender_id = weather_agent.agent_id
- receiver_id = weather_agent.agent_id
- message는 outbox에 들어감
📌 아직 처리 ❌
📌 아직 inbox ❌



3️⃣ Coordinator가 메시지를 라우팅 (전달만 함)
    coordinator.route_message()
✔️ 네 코드에 정확히 있음

Coordinator가 하는 일:
    Agent A의 outbox 확인
    → message.receiver_id 확인
    → 해당 Agent inbox로 이동
📌 중요한 포인트:
❌ 질문 해석 안 함
❌ Agent 선택 안 함
❌ 판단 안 함
👉 Message Dispatcher 역할만 수행



4️⃣ 날씨 Agent inbox에 메시지 도착
weather_agent._inbox = [Message(...)]
✔️ 네 코드 구조상 정확함

📌 Agent는 누가 보냈는지 몰라도 됨
📌 그냥 메시지가 “도착”했을 뿐



5️⃣ Coordinator가 모든 Agent에게 “일 시작” 명령
coordinator.process_all_agents()
✔️ 네 코드에 있음

실제로는:
    for agent in self.agents.values():
        agent.process_inbox()

📌 Coordinator는 “다음 Agent 실행해!” 가 아니라
“각자 받은 거 있으면 처리해!”



6️⃣ 날씨 Agent의 process_inbox() 실행
def process_inbox(self):
    self.set_state(PROCESSING)
    for message in self._inbox:
        result = self._handle_message(message)
    self._inbox = []
    self.set_state(COMPLETED)

✔️ Agent 생명주기의 핵심
이 시점에:
- 상태: IDLE → PROCESSING
- 메시지 하나씩 처리
- 처리 끝나면 inbox 비움
- 상태: COMPLETED



7️⃣ WeatherAgent의 _handle_message() 실행
    class WeatherAgent(SpecializedAgent):
        def _handle_message(self, message):
            return {"result": "서울은 비가 옵니다"}
📌 이 클래스는 네 코드에 없음
👉 (❌ 예시용, 나중에 구현할 특화 Agent)
하지만 구조적으로 실행되는 위치는 정확함
✔️ 다형성 (polymorphism)
✔️ 부모는 호출만 함
✔️ 자식이 실제 로직 수행



8️⃣ 처리 결과 반환 (하지만 저장/출력은 안 됨)
    results = agent.process_inbox()
✔️ 결과는 리스트로 반환됨
❌ 하지만 그걸 쓰는 코드가 없음
📌 그래서 지금 코드에서는:
- 결과를 UI로 출력 ❌
- 사용자에게 응답 ❌
- 다음 Agent로 전달 ❌
👉 이건 후처리 단계 (❌ 코드에 없음)



9️⃣ “서울은 비가 옵니다” 출력
📌 네 코드에는 자동 출력 없음
이걸 하려면:
- process_all_agents() 결과 수집
- print / return / API response 필요
👉 (❌ 네 코드에 없음 / 나중에 구현)




🔥 전체 흐름 한 줄 요약 (네 코드 기준)
[외부 입력 ❌]
→ [결정 로직 ❌]
→ Agent.send_message() ✅
→ Coordinator.route_message() ✅
→ Agent.receive_message() ✅
→ Agent.process_inbox() ✅
→ _handle_message() (자식) ✅
→ 결과 생성 ❌ 출력 ❌
```


In [ ]:
from enum import Enum
from dataclasses import dataclass
from typing import Dict, Any, List, Optional
from datetime import datetime
import uuid
import json

# 에이전트 상태
class AgentState(Enum):  # Enum :시스템 설계용으로 Agent가 지금 뭐하고있는지 '정해진 값'으로만 표현 / 메시지를 dict가 아니고 객체로 만들어짐
    IDLE = 'idle'   # 대기중 - 작업 할당 대기
    PROCESSING = 'processing' # 작업중 - LLM 호출중 / 다른메시지를 막을 수도 있고, 병렬처리여부를 결정할수있음
    COMPLETED = 'completed' # 결과 생성완료
    ERROR = 'error'


# 데이터클래스
@dataclass  # 데코레이터. 이걸 사용함으로써, __init__이 자동생성되어서 self.message_id = message_id 이런거 매번 안써도됨.
            # 이렇게 정의해서 사용은 가능함
            # def eileen(cls):
            #    cls.version = "1.0"
            #    return cls

            # @eileen
            # class ValuesDefinition:
            #     pass
            # print(ValuesDefinition.version)  --> 출력 1.0
class Message:   # Agent 들이 직접 말안하고 메시지로만 소통
    '''에이전트 간 메세지'''
    message_id : str
    sender_id : str
    receiver_id : Optional[str]
    content : Dict[str, Any]
    timestamp: str
    def to_dict(self) -> Dict[str,Any]:
        return{
            'id': self.message_id,    # 메시지 추적용
            'sender': self.sender_id,  # 누가 보냈는지
            'receiver' : self.receiver_id,  #누구에게
            'content' : self.content,  # 실제 작업내용
            'timestamp' : self.timestamp  # 언제 보냈는지
        }
    
class SpecializedAgent :  # 모든 Agent의 부모 클래스
    '''특화된 에이전트'''
    def __init__ (self, name:str, speciality:str):
        '''
        Args:
            name : 에이전트 이름
            speciality : 전문분야
        '''
        self.agent_id = str(uuid.uuid4())[:8]
        self.name = name
        self.speciality = speciality
        self._state = AgentState.IDLE  # 아무일 안하는 상태에서 시작
        self._inbox : List[Message] = []  # 받은 메시지
        self._outbox : List[Message] = []  # 보낼 메시지
    
    def receive_message(self, message: Message):  # 메시지 받기: 그냥 inbox안에 쌓음. 처리 안함
        '''메세지 수신'''
        self._inbox.append(message)
    
    def send_message(self, receiver_id:str, content:Dict[str, Any]): # Message에 객체 생성, outbox에 넣음. 실제 전달은 coordinator가 함. Agent는 "보낸다"만 하지 "전달"은 안함
        '''메세지 전송'''
        message=Message(
            message_id=str(uuid.uuid4())[:8],
            sender_id = self.agent_id,
            receiver_id=receiver_id,
            content=content,
            timestamp=datetime.now().isoformat()
        )
        self._outbox.append(message)
        return message
    
        # process_inbox()
            # 🧠 이게 왜 중요한가?
            # process_inbox()는 아무 것도 모른다
            # 이게 수학인지
            # 텍스트인지
            # 벡터 검색인지
            # 그냥 이렇게만 함 👇
            # result = self._handle_message(message)
            # 📌 “어떻게 처리하는지는 네가 알아서 해”
            # 👉 이게 Agent 설계의 핵심 철학
    def process_inbox(self) -> list[Dict[str,Any]]:  # Agent가 일하는 순간 ==> 이 함수 하나가 Agent의 생명 주기 / 이 함수가 호출될때 마다 agent는 일을 시작, 메시지를 처리, 결과를 생성, 상태를 완료로 변경
        '''받은 메세지 처리'''
        self.set_state(AgentState.PROCESSING) # self._state = self.set_state(AgentState.PROCESSING) 이렇게 기재하면, set_state return 값이 없으므로..? 받으면 안됨..?? 무슨말
        results = []
        for message in self._inbox:
            result = self._handle_message(message)
            results.append(result)
        self._inbox = [] # 상기 처리된 메세지를 제거함/ 중복처리 방지
        self.set_state(AgentState.COMPLETED)
        return results
    
    def _handle_message(self, message:Message) -> Dict[str, Any]: # 부모 클래스
        '''메세지 처리(오버라이드 가능 (다시 구현 가능?))'''
        return{
            'status':'handled',
            'message_id': message.message_id,
            'content': message.content
        }
    

        # 게터/세터 왜써?
            # 상태를 마음대로 바꾸지 못하게 막는 안전장치
            # agent._state = "hello" 이렇게 직접 접근시 상태 망가지고, 규칙 깨짐
            # agent.set_state(AgentState.PROCESSING) --> 정해진 Enum만 허용, 규칙통제
    def get_state(self) -> str: 
        return self._state.value
        # get_state: 상태를 "읽기 전용 문자열"로 제공 / 값반환이므로 매개변수 필요없음/ return값음 있음
        # get_state: 상태가 궁금해? 보기좋은 문자열만 보여줄게
    def set_state(self, state:AgentState):  
        self._state = state
        # set_state: 상태를 "정해진 규칙(Enum)"으로만 변경 / 매개변수는 있으나, return값이 없음.
        # set_state: 상태를 바꾸고 싶어? AgentState 중 하나만 가져와
    def get_info(self) -> Dict[str, Any]:
        '''에이전트의 상태를 반환'''
        return{
            'id': self.agent_id,
            'name': self.name,
            'speciality': self.speciality,
            'state': self.get_state(),
            'inbox_size': len(self._inbox),
            'outbox_size' : len(self._outbox)
        }
        

# 누가 보낸메시지를 누구에게 전달할지 정리하는 역할 / 오로지 전달만!
# Transport Router (전달자) : "똑똑한 Agent"가 아니라 택배물류센터 (목적지는 이미 정해져있고, 박스만 정확히 전달)
# class Coordinator는 분기를 결정하는 Decision Router가 아니다!!! 이건 지금 코드에 빠져있음
    # 실제하는일
    # Agent A outbox
    #    ↓
    # Coordinator
    #    ↓
    # Agent B inbox
    # 📌 Agent는 서로 몰라
    # 📌 Coordinator만 앎 
class Coordinator:
    '''에이전트 조정자 (메세지 라우터)'''
    def __init__(self):
        self.agents : Dict[str, SpecializedAgent] = {}
    def register_agent(self, agent:SpecializedAgent):
        '''에이전트 등록'''
        self.agents[agent.agent_id] = agent

        # ✔️ 이미 receiver_id가 정해진 메시지를 ✔️ 해당 Agent inbox로 옮겨주는 역할만 함
        # Transport Router (전달자) 에 해당함!!
    def route_message(self):
        '''모든 에이전트의 메세지를 라우팅'''
        for agent in self.agents.values() : # 모든 agents의 values --> SpecializedAgent
            for message in agent._outbox: # 수신자 에이전트의 정보 등등
                if message.receiver_id in self.agents :  # 수신자와 송신자를 매칭 # validation 체크. 모든 에이전트의 id중에서 수신자 에이전트가 있으면.
                    receiver = self.agents[message.receiver_id]
                    receiver.receive_message(message)
                    print(f"[OK] {message.message_id} : {agent.name} ---> {receiver.name}")

    def process_all_agents(self):
        '''모든 에이전트의 메세지를 처리'''
        for agent in self.agents.values():
            agent.process_inbox()


    def system_status(self):
        '''시스템의 상태를 출력'''
        for agent in self.agents.values():
            print(agent.get_info())


# 특화된 에이전트 구현
class TextProcessorAgent(SpecializedAgent):
    '''텍스트 처리 에이전트'''
    def _handle_message(self, message) -> Dict[str,Any]: 
        content = message.content
        operation = content.get('operation', '')
        text = content.get('text', '')
        if operation == 'uppercase':
            result = text.upper()
        elif operation == 'lowercase':
            result = text.lower()
        elif operation == 'reverse':
            result = text[::-1]
        else:
            result = text
        
        return{
            'status':'processed',
            'operation': operation,
            'input': text,
            'output': result
        }


class MathAgent(SpecializedAgent):
    def _handle_message(self, message:Message) -> Dict[str,Any]:
        content = message.content
        operation = content.get('operation', '')
        a=content.get('a', 0)
        b=content.get('b', 0)
        if operation == 'subtract':
            result = a-b
        elif operation == 'add':
            result = a+b
        elif operation == 'multiply':
            result = a*b
        elif operation == 'divide':
            result = a / b if b !=0 else 0
        else:
            result = 0
        return {
            'status' : 'calculated',
            'a' : a,
            'b' : b,
            'result' : result
        }
    


class DataAnalyzerAgent(SpecializedAgent):
    """데이터 분석 에이전트"""
    
    def _handle_message(self, message: Message) -> Dict[str, Any]:
        content = message.content
        data = content.get("data", [])
        
        print(f"   {self.name}: {len(data)}개 항목 분석")
        
        if isinstance(data, list) and len(data) > 0:
            if all(isinstance(x, (int, float)) for x in data): # isinstance(x, (int, float)) int or float 둘중 하나만 있어도 True
                avg = sum(data) / len(data)
                max_val = max(data)
                min_val = min(data)
                
                return {
                    "status": "analyzed",
                    "count": len(data),
                    "average": avg,
                    "max": max_val,
                    "min": min_val
                }
        
        return {"status": "invalid_data"}
    


if __name__ == '__main__':
    text_agent = TextProcessorAgent('textbot', 'text processing')
    math_agent = MathAgent('mathbot', 'calculate')
    analyzer_agent = DataAnalyzerAgent('analyzerbot', 'data analysis')

    # 코디네이터(조정자)에 등록
    coordinator = Coordinator()
    coordinator.register_agent(text_agent)
    coordinator.register_agent(math_agent)
    coordinator.register_agent(analyzer_agent)

    # 메서드
    text_agent.send_message(
        text_agent.agent_id,
        {'operation' : 'uppercase', 'text' : 'hong-gil-dong'}
    )
    math_agent.send_message(
        math_agent.agent_id,
        {'operation': 'multiply', 'a':10, 'b':2}
    )
    analyzer_agent.send_message(
        analyzer_agent.agent_id,
        {'data': [1,2,3,4,5]}
    )

    # 메세지 라우팅 : agent_id에 해당하는 메세지를 해당 agent의 _inbox에 저장
    coordinator.route_message()

    # 메세지 처리
    coordinator.process_all_agents()

    # 시스템 상태 출력
    coordinator.system_status()

[OK] fce02074 : textbot ---> textbot
[OK] a8352187 : mathbot ---> mathbot
[OK] 5987510d : analyzerbot ---> analyzerbot
   analyzerbot: 5개 항목 분석
{'id': '4865d878', 'name': 'textbot', 'speciality': 'text processing', 'state': <AgentState.COMPLETED: 'completed'>, 'inbox_size': 0, 'outbox_size': 1}
{'id': '92c9b07c', 'name': 'mathbot', 'speciality': 'calculate', 'state': <AgentState.COMPLETED: 'completed'>, 'inbox_size': 0, 'outbox_size': 1}
{'id': 'c72fc1d4', 'name': 'analyzerbot', 'speciality': 'data analysis', 'state': <AgentState.COMPLETED: 'completed'>, 'inbox_size': 0, 'outbox_size': 1}
